In [13]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [14]:
set_seed(seed=42)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [16]:
df = pd.read_excel(
    "../../../data/bpic20_Dom.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Amount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [17]:
df.head(20)

,case:concept:name,time:timestamp,case:Amount,concept:name,org:resource,org:role,time_delta
0,declaration 100000,2018-01-30 09:20:07,600.844116,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 100000,2018-02-07 09:58:46,600.844116,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,693519.0
2,declaration 100000,2018-02-08 10:59:05,600.844116,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,90019.0
3,declaration 100000,2018-02-09 12:42:49,600.844116,Request Payment,SYSTEM,UNDEFINED,92624.0
4,declaration 100000,2018-02-12 17:31:20,600.844116,Payment Handled,SYSTEM,UNDEFINED,276511.0
5,declaration 100005,2018-01-30 09:38:54,35.133686,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
6,declaration 100005,2018-01-30 09:38:57,35.133686,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0
7,declaration 100005,2018-01-30 10:04:10,35.133686,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,1513.0
8,declaration 100005,2018-01-31 12:45:18,35.133686,Request Payment,SYSTEM,UNDEFINED,96068.0
9,declaration 100005,2018-02-01 17:31:17,35.133686,Payment Handled,SYSTEM,UNDEFINED,103559.0


In [18]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Amount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [2.00, 284910.00]                        42280.0000 quantile_derived    
case:Amount                    continuous     case     yes    [6.91, 219.03]                           25.3942    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
org:role                       categorical    event    ye

In [19]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [20]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [21]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [22]:
scenario_df.head()

,case:concept:name,time_index,fake,case:Amount,concept:name,org:resource,org:role,time_delta
0,declaration 131033,0,False,6.36031,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 131033,1,False,6.36031,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,96.0
2,declaration 131033,2,False,6.36031,Declaration APPROVED by BUDGET OWNER,STAFF MEMBER,BUDGET OWNER,171818.0
3,declaration 131033,3,False,6.36031,Declaration REJECTED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,491478.0
4,declaration 131033,4,False,6.36031,Declaration REJECTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,94929.0


In [23]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [24]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [25]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
criterion = torch.nn.BCEWithLogitsLoss()

In [28]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Dom-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0111 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0103 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0093 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0079 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0074 | LR: 1.00e-06
Time taken for scenario model (training): 2579.223577 seconds
Time taken for scenario model (validation): 2.229346 seconds
Val loss: {'loss': 0.009434661733671882, 'accuracy': 0.9976271261614885, 'f1_macro': 0.9968773483084812, 'f1_weighted': 0.9976304855028827}


In [29]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [30]:
# scenario_model = ScenarioLSTM.load()

In [31]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [32]:
sys.stdout = original_stdout
log_file.close()